Sistem Rekomendasi Anime

# Data Understanding

In [1]:
# Import library
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load data
!pip install kaggle

from google.colab import files
files.upload()

!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d CooperUnion/anime-recommendations-database
!unzip anime-recommendations-database.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database
License(s): CC0-1.0
 72% 18.0M/25.0M [00:00<00:00, 80.1MB/s]
100% 25.0M/25.0M [00:00<00:00, 89.1MB/s]
Archive:  anime-recommendations-database.zip
  inflating: anime.csv               
  inflating: rating.csv              


## Variabel Anime


In [3]:
# Membuat dataset bernama anime
anime = pd.read_csv('anime.csv')
anime.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


- `anime_id`: Id unik untuk judul anime
- `name`: Nama anime
- `genre`: Genre anime
- `type`: Tipe seperti Movie, TV, OVA, dll
- `episodes`: Jumlah episode
- `rating`: rating untuk anime
- `members`: jumlah anggota komunitas anime tersebut


In [4]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [5]:
anime.shape

(12294, 7)

Pada data `anime` terdapat 7 kolom dengan 12.294 baris data

In [6]:
# Melihat jumlah genre dan apa saja genrenya
print('Banyak genre anime: ', len(anime.genre.unique()))
print('genre: ', anime.genre.unique())

Banyak genre anime:  3265
genre:  ['Drama, Romance, School, Supernatural'
 'Action, Adventure, Drama, Fantasy, Magic, Military, Shounen'
 'Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen' ...
 'Hentai, Sports' 'Drama, Romance, School, Yuri' 'Hentai, Slice of Life']


Jumlah genre pada data `anime` adalah 3265 data

In [7]:
# Cek missing value
anime.isnull().sum()

,0
anime_id,0
name,0
genre,62
type,25
episodes,0
rating,230
members,0


Pada data `anime` terdapat missing value pada beberapa fitur sebagai berikut:
- `genre`: 62 data
- `type`: 25 data
- `rating`: 230 data

missing value ini akan kita hapus nanti saat tahap preparation

## Variabel Rating

In [8]:
# Membuat dataset bernama ratings
ratings = pd.read_csv('rating.csv')
ratings.head()

,user_id,anime_id,rating
0,1,20,-1
1,1,24,-1
2,1,79,-1
3,1,226,-1
4,1,241,-1


- `user_id`: Id user.
- `anime_id`: id anime.
- `rating`: Rating dari user (jika bernilai -1 berarti user hanya menonton dan tidak memberi rating).


In [9]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7813737 entries, 0 to 7813736
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 178.8 MB


In [10]:
ratings.shape

(7813737, 3)

Pada data `ratings` terdapat 3 kolom dengan 7.813.737 baris data

In [11]:
# Cek missing value
ratings.isnull().sum()

,0
user_id,0
anime_id,0
rating,0


Tidak terdapat missing value pada data `ratings`

In [12]:
# cek jumlah data yang ratingnya -1
ratings[ratings.rating == -1].shape

(1476496, 3)

Terdapat 1.476.496 data yang memiliki `rating` bernilai -1 atau bisa dibilang sebanyak 1.476.496 user hanya menonton tanpa memberikan rating terhadap anime tersebut

# Data Preparation

## Mengatasi Missing Value pada data `anime`

In [13]:
# Menghapus baris dengan missing value pada kolom 'genre', 'type', dan 'rating'
anime.dropna(subset=['genre', 'type', 'rating'], inplace=True)

# Cek kembali missing value
anime.isnull().sum()

,0
anime_id,0
name,0
genre,0
type,0
episodes,0
rating,0
members,0


Missing value pada data `anime` berhasil dihapus

## Menghapus rating bernilai -1 pada data `ratings`

In [14]:
# Menghapus baris dengan rating -1
ratings = ratings[ratings.rating != -1]

# Cek kembali data
ratings.head()
ratings.shape

(6337241, 3)

`rating` bernilai -1 pada data `rating` berhasil dihapus dan sisa data sekarang adalah 6.337.241 baris data

## Menggabungkan data anime dan ratings

In [15]:
# Menggabungkan data
df = pd.merge(anime, ratings, on='anime_id', how='inner')
df.head()

,anime_id,name,genre,type,episodes,rating_x,members,user_id,rating_y
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,99,5
1,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,152,10
2,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,244,10
3,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,271,10
4,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,322,10


Penggabungan data berhasil
karena sebelumnya terdapat 2 fitur `rating` pada masing-masing data, ootomatis namanya berubah mennjadi:
- `rating_x`: rating keseluruhan anime
- `rating_y`: rating yang diberikan user

In [16]:
df.shape

(6337146, 9)

Jumlah data setelah di merge menjadi 9 kolom dengan total 6.337.146 baris data

# Mengatasi duplicate data fitur `name`

In [17]:
# Cek jumlah duplicate data pada fitur 'name'
df.name.value_counts()

,count
name,
Death Note,34226
Sword Art Online,26310
Shingeki no Kyojin,25290
Code Geass: Hangyaku no Lelouch,24126
Angel Beats!,23565
...,...
Omakase! Miracle Cat-dan,1
Omakase Scrappers,1
Okore!! Nonkuro,1


In [18]:
# Hapus duplicate data pada fitur 'name' lalu memasukkannya pada variabel baru bernama 'preparation'
preparation = df.drop_duplicates('name')
preparation

,anime_id,name,genre,type,episodes,rating_x,members,user_id,rating_y
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630,99,5
1961,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665,3,10
23455,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262,43,10
24643,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572,5,9
41794,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266,5,9
...,...,...,...,...,...,...,...,...,...
6337138,5541,The Satisfaction,Hentai,OVA,1,4.37,166,37351,1
6337140,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai,OVA,1,4.15,211,20171,7
6337142,5543,Under World,Hentai,OVA,1,4.28,183,49503,4
6337144,5621,Violence Gekiga David no Hoshi,Hentai,OVA,4,4.88,219,49503,6


Setelah menghapus semua duplicate pada baris `name`, sisa data menjadi 9892 baris

In [19]:
# Cek kembali jumlah duplicate data pada fitur 'name'
preparation.name.value_counts()

,count
name,
Kimi no Na wa.,1
Dark Side Cat,1
Marie &amp; Gali Episode Zero,1
Mobile Suit SD Gundam Mk V,1
Mother: Saigo no Shoujo Eve,1
...,...
Captain Tsubasa: Saikyou no Teki! Holland Youth,1
Chou Hatsumei Boy Kanipan,1
Coo: Tooi Umi kara Kita Coo,1


Sudah tidak ada duplicate data pada variabel `preparation`

## Membuat dictionary pada fitur `anime_id`, `name`, dan `genre`

In [20]:
# Melakukan konversi data series menjadi list
anime_id = preparation['anime_id'].tolist()
name = preparation['name'].tolist()
genre = preparation['genre'].tolist()

print(len(anime_id))
print(len(name))
print(len(genre))

9892
9892
9892


In [21]:
# Membuat dictionary lalu memasukkannya pada variabel 'df_new"
df_new = pd.DataFrame({
    'anime_id': anime_id,
    'name': name,
    'genre': genre
})
df_new

,anime_id,name,genre
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural"
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili..."
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S..."
3,9253,Steins;Gate,"Sci-Fi, Thriller"
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S..."
...,...,...,...
9887,5541,The Satisfaction,Hentai
9888,9316,Toushindai My Lover: Minami tai Mecha-Minami,Hentai
9889,5543,Under World,Hentai
9890,5621,Violence Gekiga David no Hoshi,Hentai


Data `df_new` sudah siap masuk ke dalam permodelan

# Modelling

## Model Development dengan Content Based Filtering

In [22]:
# Menampilkan 5 data random
df_new.sample(5)

,anime_id,name,genre
3405,2646,Dorami &amp; Doraemons: Robot School&#039;s Se...,Fantasy
1252,19195,Ghost in the Shell: Arise - Border:4 Ghost Sta...,"Mecha, Police, Psychological, Sci-Fi"
6581,21081,Yu Bang Xiang Zheng,Historical
1740,33302,Yowamushi Pedal: Spare Bike,"Comedy, Drama, Shounen, Sports"
7726,28645,Haru no Shikumi,Dementia


In [23]:
# Inisialisasi TfidfVectorizer
tf = TfidfVectorizer()

# Melakukan perhitungan idf pada data genre
tf.fit(df_new['genre'])

# Mapping array dari fitur index integer ke fitur name
tf.get_feature_names_out()

array(['action', 'adventure', 'ai', 'arts', 'cars', 'comedy', 'dementia',
       'demons', 'drama', 'ecchi', 'fantasy', 'fi', 'game', 'harem',
       'hentai', 'historical', 'horror', 'josei', 'kids', 'life', 'magic',
       'martial', 'mecha', 'military', 'music', 'mystery', 'of', 'parody',
       'police', 'power', 'psychological', 'romance', 'samurai', 'school',
       'sci', 'seinen', 'shoujo', 'shounen', 'slice', 'space', 'sports',
       'super', 'supernatural', 'thriller', 'vampire', 'yaoi', 'yuri'],
      dtype=object)

In [24]:
# Melakukan fit lalu ditransformasikan ke bentuk matrix
tfidf_matrix = tf.fit_transform(df_new['genre'])

# Melihat ukuran matrix tfidf
tfidf_matrix.shape

(9892, 47)

Nilai 9882 merupaka ukuran data, sedangkan 47 adalah matriks ukuran genre

In [25]:
# Mengubah vektor tf-idf dalam bentuk matriks dengan fungsi todense()
tfidf_matrix.todense()

matrix([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.29047501, 0.32071235, 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.24385925, 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
         0.        ]])

In [26]:
# Membuat dataframe untuk melihat tf-idf matrix
# Kolom diisi dengan genre
# Baris diisi dengan nama anime

num_cols_to_sample = min(47, len(tf.get_feature_names_out()))  # Ensure sample size <= number of columns
num_rows_to_sample = min(10, tfidf_matrix.shape[0]) # Ensure sample size <= number of rows

pd.DataFrame(
    tfidf_matrix.todense(),
    columns=tf.get_feature_names_out(),
    index=df_new.name
).sample(num_cols_to_sample, axis=1, replace=False).sample(num_rows_to_sample, axis=0, replace=False)

,of,super,harem,romance,mystery,dementia,shoujo,life,demons,supernatural,...,historical,slice,seinen,psychological,game,samurai,yaoi,adventure,fi,kids
name,,,,,,,,,,,,,,,,,,,,,
Kishin Houkou Demonbane (TV) Specials,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.472300,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.373956,0.000000
Ahiru no Pekkle no Minikui Ahiru no Ko,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.796884
Golden Time,0.0,0.0,0.0,0.553015,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.748834,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
Shokichi Monogatari,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.698331
Kuroshitsuji Picture Drama,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.807629,0.589691,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
Kibun Kibun,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
I&#039;&#039;s,0.0,0.0,0.0,0.728635,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
Sailor Fuku Shinryou Tsumaka,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
Mahou no Princess Minky Momo vs. Mahou no Tenshi Creamy Mami,0.0,0.0,0.0,0.000000,0.0,0.0,0.721302,0.0,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000


In [27]:
# Menghitung cosine similarity pada matrix tf-idf
cosine_sim = cosine_similarity(tfidf_matrix)
cosine_sim

array([[1.        , 0.15439041, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.15439041, 1.        , 0.17128271, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.17128271, 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 1.        ,
        1.        ],
       [0.        , 0.        , 0.        , ..., 1.        , 1.        ,
        1.        ],
       [0.        , 0.        , 0.        , ..., 1.        , 1.        ,
        1.        ]])

In [28]:
# Membuat dataframe dari variabel cosine_sim dengan baris dan kolom berupa nama anime
cosine_sim_df = pd.DataFrame(cosine_sim, index=df_new['name'], columns=df_new['name'])
print('Shape:', cosine_sim_df.shape)

# Melihat similarity matrix pada setiap nama anime
cosine_sim_df.sample(5, axis=1).sample(10, axis=0)

Shape: (9892, 9892)


name,Mou Hitotsu no Nijiiro Toshi,Mahou Shoujo? Naria☆Girls,Sora no Method: Aru Shoujo no Kyuujitsu★,Haha wo Tazunete Sanzenri Specials,Ao no Exorcist Movie Special
name,,,,,
Dororon Enma-kun Meeramera,0.0,0.384870,0.173808,0.000000,0.083846
Piano no Mori,0.0,0.000000,0.076156,0.278387,0.104934
Kunoichi Gakuen Ninpouchou,0.0,0.000000,0.058968,0.000000,0.081250
Kagachi-sama Onagusame Tatematsurimasu: Netorare Mura Inya Hanashi The Animation,0.0,0.000000,0.000000,0.000000,0.000000
Fate/Prototype,0.0,0.497811,0.146103,0.000000,0.000000
Denshinbashira no Okaasan,0.0,0.000000,0.000000,0.404518,0.000000
Mamoru-kun ni Megami no Shukufuku wo!,0.0,0.000000,0.120977,0.000000,0.166690
Slam Dunk: Hoero Basketman-damashii! Hanamichi to Rukawa no Atsuki Natsu,0.0,0.000000,0.620616,0.660254,0.089329
Ontama!,0.0,0.450110,0.681265,0.586273,0.098059


In [32]:
def anime_recommendations(nama_anime, similarity_data=cosine_sim_df, items=df_new[['name', 'genre']], k=5):
    index = similarity_data.loc[:,nama_anime].to_numpy().argpartition(
        range(-1, -k, -1))

    # Mengambil data dengan similarity terbesar dari index yang ada
    closest = similarity_data.columns[index[-1:-(k+2):-1]]

    # Drop nama_anime agar nama resto yang dicari tidak muncul dalam daftar rekomendasi
    closest = closest.drop(nama_anime, errors='ignore')

    return pd.DataFrame(closest).merge(items).head(k)

In [44]:
# Menampilkan data pada anime `Detective Conan OVA 11: A Secret Order from London`

df_new[df_new.name.eq('Detective Conan OVA 11: A Secret Order from London')]

,anime_id,name,genre
1493,10703,Detective Conan OVA 11: A Secret Order from Lo...,"Adventure, Comedy, Mystery, Police, Shounen"


In [43]:
# Menampilkan 5 rekomendasi
anime_recommendations('Detective Conan OVA 11: A Secret Order from London')

,name,genre
0,Detective Conan OVA 03: Conan and Heiji and th...,"Adventure, Comedy, Mystery, Police, Shounen"
1,Detective Conan Movie 01: The Timed Skyscraper,"Adventure, Comedy, Mystery, Police, Shounen"
2,Aoyama Goushou Tanpenshuu,"Adventure, Comedy, Mystery, Police, Shounen"
3,Detective Conan Movie 04: Captured in Her Eyes,"Adventure, Comedy, Mystery, Police, Shounen"
4,Detective Conan: Black History 2,"Adventure, Comedy, Mystery, Police, Shounen"


Rekomendasi 5 anime baru yang sesuai dengan `Detective Conan OVA 11: A Secret Order from London` sudah muncul



# Evaluation

In [42]:
# Evaluasi model
Tp = 5
Fp = 0

Precision = (Tp / (Tp + Fp))*100
print(Precision)

100.0


- Precision mengukur proporsi prediksi positif yang benar dibandingkan dengan semua prediksi positif.
- Tp = True positive (Jumlah prediksi positif yang benar)
- Fp = False positive (jumlah prediksi negatif yang benar)

Berdasarkan pada precision evaluasinya, didapatkan nilai 100% pada precisionnya. Ini menandakan prediksi benar semua